# ToolCallingAgent in DemoGPT

The **ToolCallingAgent** is a single-tool calling agent designed for straightforward tasks. Given a user query, it reasons about which tool is best suited to answer the question, calls that tool exactly once, and returns a synthesized result.

This makes it ideal for tasks where a single tool invocation is sufficient -- such as looking something up, running a calculation, or converting a value. It is faster and simpler than the multi-step `ReactAgent`, which is better suited for complex, multi-step reasoning.

## Setting Up Your API Key

Create a `.env` file in your project root with your OpenAI API key:

```
OPENAI_API_KEY=sk-your-key-here
```

Then load it with `python-dotenv` (already included with DemoGPT):

In [ ]:
!pip install demogpt

In [ ]:
from dotenv import load_dotenv
load_dotenv("../.env")  # Loads OPENAI_API_KEY from .env file

import os
print("API key loaded:", "OPENAI_API_KEY" in os.environ)

## 2. How ToolCallingAgent Works

The ToolCallingAgent follows a simple, linear flow:

1. **Receives a query** from the user
2. **Reasons** about which tool from its available tools is best suited to answer the query
3. **Calls the selected tool** with the appropriate arguments
4. **Synthesizes the answer** by combining the tool's output with the original query

When `verbose=True`, the agent prints each step of this process:

- **Reasoning**: The agent's thought process about which tool to use and why
- **Tool call**: The name of the selected tool
- **Tool result**: The raw output from the tool
- **Answer**: The final synthesized response to the user

## 3. Basic Example - Web Search

Let's start with a simple example: giving the agent a single search tool and asking it a question about current events.

In [ ]:
from demogpt_agenthub.agents import ToolCallingAgent
from demogpt_agenthub.tools import TavilySearchTool
from demogpt_agenthub.llms import OpenAIChatModel

# Initialize the language model
llm = OpenAIChatModel(model_name="gpt-4o-mini")

# Create a search tool
search = TavilySearchTool()

# Create the agent with the search tool
agent = ToolCallingAgent(tools=[search], llm=llm, verbose=True)

# Run the agent
response = agent.run("What are the latest developments in quantum computing?")
print(response)

## 4. Multi-Tool Agent

The real power of ToolCallingAgent shows when you give it multiple tools. The agent will automatically reason about which tool is the best fit for the query and select it.

Here we provide three tools:
- **TavilySearchTool** - for web searches
- **WikipediaTool** - for factual/encyclopedic questions
- **PythonTool** - for math and code execution

In [ ]:
from demogpt_agenthub.tools import TavilySearchTool, WikipediaTool, PythonTool

# Create multiple tools
search = TavilySearchTool()
wikipedia = WikipediaTool()
python = PythonTool()

# Create agent with all three tools
multi_agent = ToolCallingAgent(
    tools=[search, wikipedia, python],
    llm=llm,
    verbose=True
)

In [ ]:
# Math question - the agent should pick the PythonTool
response = multi_agent.run("What is the square root of 1764 plus 42 squared?")
print(response)

In [ ]:
# Factual question - the agent should pick WikipediaTool
# Note: ToolCallingAgent creates a new instance for each independent query
multi_agent2 = ToolCallingAgent(
    tools=[search, wikipedia, python],
    llm=llm,
    verbose=True
)
response = multi_agent2.run("Who was Marie Curie and what did she discover?")
print(response)

## 5. Verbose Mode

The `verbose` parameter controls whether the agent prints its internal reasoning process. When set to `True`, the agent displays color-coded output showing each step:

- The **reasoning** behind tool selection
- Which **tool** was called
- The **raw result** from the tool
- The final **answer** synthesized from the result

Setting `verbose=False` suppresses all intermediate output, and the agent silently returns just the final answer. This is useful in production or when you only need the result.

In [ ]:
# Verbose mode ON - shows reasoning, tool call, tool result, and answer
print("=== verbose=True ===")
verbose_agent = ToolCallingAgent(tools=[search], llm=llm, verbose=True)
response = verbose_agent.run("What is the population of Tokyo?")
print(f"\nFinal response: {response}")

In [ ]:
# Verbose mode OFF - only the final answer is returned, no intermediate output
print("=== verbose=False ===")
quiet_agent = ToolCallingAgent(tools=[search], llm=llm, verbose=False)
response = quiet_agent.run("What is the population of Tokyo?")
print(f"Final response: {response}")

## 6. Custom Tools with ToolCallingAgent

You can create your own tools by extending `BaseTool`. A custom tool needs:
- A `name` attribute (used by the agent to identify the tool)
- A `description` attribute (used by the agent to decide when to use the tool)
- A `run` method that takes input and returns a result

Below, we create a simple temperature unit converter tool and use it alongside the search tool.

In [ ]:
from demogpt_agenthub.tools import BaseTool

class UnitConverterTool(BaseTool):
    def __init__(self):
        self.name = "UnitConverter"
        self.description = "Converts temperatures between Celsius and Fahrenheit. Input format: '32F to C' or '100C to F'"
        super().__init__()

    def run(self, query: str):
        # Simple temperature conversion
        query = query.strip().upper()
        if "F TO C" in query:
            f = float(query.split("F")[0])
            return f"{(f - 32) * 5/9:.2f} Celsius"
        elif "C TO F" in query:
            c = float(query.split("C")[0])
            return f"{c * 9/5 + 32:.2f} Fahrenheit"
        return "Invalid format. Use '32F to C' or '100C to F'"

# Create the custom tool
converter = UnitConverterTool()

# Create an agent with the custom tool and a search tool
agent = ToolCallingAgent(
    tools=[converter, TavilySearchTool()],
    llm=llm,
    verbose=True
)

# The agent should pick the UnitConverter tool for this query
response = agent.run("Convert 98.6 Fahrenheit to Celsius")
print(response)

## 7. When to Use ToolCallingAgent vs ReactAgent

DemoGPT provides two agent types. Choosing the right one depends on the complexity of your task:

| Feature | ToolCallingAgent | ReactAgent |
|---|---|---|
| **Tool calls** | Single tool call per query | Multiple tool calls in a reasoning loop |
| **Speed** | Faster (one-shot) | Slower (iterative reasoning) |
| **Complexity** | Best for straightforward tasks | Best for complex, multi-step tasks |
| **Use cases** | Lookups, calculations, conversions | Research, analysis, tasks requiring multiple sources |
| **Reasoning** | Selects the best tool once | Reasons, acts, observes, and repeats |

**Use ToolCallingAgent when:**
- The answer can be obtained from a single tool call
- You need fast, direct answers
- The task is simple and well-defined (e.g., "search for X", "calculate Y", "convert Z")

**Use ReactAgent when:**
- The task requires combining information from multiple sources
- You need iterative reasoning (e.g., "compare X and Y", "research and summarize")
- The agent may need to refine its approach based on intermediate results

## 8. Summary

In this notebook, we covered:

- **What ToolCallingAgent is**: A single-tool calling agent that selects the best tool, calls it once, and synthesizes an answer
- **Basic usage**: Creating an agent with a search tool to answer questions
- **Multi-tool agents**: Providing multiple tools and letting the agent choose the right one based on the query
- **Verbose mode**: Using `verbose=True` to inspect the agent's reasoning, tool selection, and results
- **Custom tools**: Extending `BaseTool` to create your own tools and integrating them with the agent
- **Agent comparison**: When to choose ToolCallingAgent (simple, one-shot tasks) vs ReactAgent (complex, multi-step tasks)

Next, check out **05 - React Agent** to learn about multi-step reasoning agents that can chain multiple tool calls together.